# Rerank MagicLens bằng VLM (Qwen2-VL) trên Google Colab

Notebook này chấm lại **top-K ứng viên của MagicLens** trên FashionIQ dress bằng một mô hình
thị giác–ngôn ngữ (VLM). Với mỗi cặp (ảnh truy vấn, ảnh ứng viên), VLM được hỏi:
*"ảnh ứng viên có phải là ảnh truy vấn sau khi sửa theo câu chỉ dẫn không?"*, và điểm là
`logit("Yes") − logit("No")`. Phần trộn điểm và đánh giá chạy trên máy cá nhân bằng
`vlm_rerank_eval.py`. Hướng dẫn đầy đủ: `docs/huong_dan_vlm_rerank.md`.

**Trước khi chạy:**
1. Trên máy cá nhân đã chạy `python vlm_rerank_prepare.py --subtask dress`, tạo ra `vlm_rerank/bundle.zip` (khoảng 290 MB).
2. Tải `bundle.zip` lên Google Drive, vào thư mục **`MyDrive/magiclens_vlm/`**.
3. Trong Colab: **Runtime → Change runtime type → T4 GPU**.

**Chạy lần lượt từng ô từ trên xuống.** Kết quả được ghi thẳng lên Drive sau mỗi truy vấn. Nếu Colab
bị ngắt, chỉ cần kết nối lại, chạy lại các ô 1–5 rồi ô 7: script sẽ chạy tiếp từ chỗ dừng.

**Mỗi khi Colab khởi động lại** (bị ngắt, mở lại notebook, Restart session), mọi biến, thư viện vừa cài và
dữ liệu đã giải nén đều mất; chỉ dữ liệu trên Drive còn lại. Vì vậy luôn chạy lại ô 1–5 trước khi chạy ô 6–9.
Nếu quên, các ô sau sẽ báo cần chạy lại ô nào.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Kết nối Google Drive
Colab sẽ hỏi quyền truy cập Drive; chọn tài khoản chứa `bundle.zip`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Cấu hình

Thường không cần sửa gì. Các trường hợp cần đổi (sửa rồi chạy lại ô này):
- Báo **CUDA out of memory**: giảm `BATCH_SIZE` (ví dụ 4).
- Báo **Non-finite scores** (float16 bị tràn số): đặt `DTYPE = "float32"` **và** đổi tên `OUT`
  (ví dụ `vlm_scores_fp32.jsonl`), vì không được trộn điểm của hai kiểu số.
- Muốn nhanh hơn: `TOP_K = 30` (ít cặp hơn 40%, nhưng trần R@10 giảm từ 42.1 xuống 35.7).
  Không đặt dưới 20: rerank một danh sách ngắn hơn không thay đổi được R@10.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/magiclens_vlm"  # thư mục chứa bundle.zip trên Drive
BUNDLE_DIR = "/content/bundle"                       # giải nén ra ổ của Colab (đọc ảnh nhanh hơn Drive)
MODEL = "Qwen/Qwen2-VL-2B-Instruct"
TOP_K = 50        # số ứng viên chấm cho mỗi truy vấn (từ 20 đến 50)
BATCH_SIZE = 10   # số cặp mỗi lượt chạy GPU (đổi giữa chừng không sao)
DTYPE = "float16" # chạy được trên mọi GPU của Colab; để cố định để chạy tiếp được dù Colab cấp GPU khác
OUT = f"{DRIVE_DIR}/vlm_scores.jsonl"  # kết quả, ghi thẳng lên Drive
NO_CACHE_MARK = f"{DRIVE_DIR}/use_no_vision_cache"  # ô 6 tạo tệp này nếu phải dùng cách chạy dự phòng

assert os.path.exists(f"{DRIVE_DIR}/bundle.zip"), (
    f"Không thấy {DRIVE_DIR}/bundle.zip: chạy ô 2 (kết nối Drive) và kiểm tra đã tải bundle.zip "
    "lên đúng thư mục MyDrive/magiclens_vlm.")
print("OK, kết quả sẽ ghi vào", OUT)

## 4. Cài thư viện

Colab đã có sẵn PyTorch và torchvision; chỉ cần cài transformers đúng bản đã kiểm thử (4.57.6) và accelerate.
Pip có thể in một đoạn bắt đầu bằng `ERROR: pip's dependency resolver does not currently take into account...`:
đó chỉ là cảnh báo xung đột phiên bản, **bỏ qua được** nếu dòng cuối của ô hiện `transformers 4.57.6` và `GPU: True`.

In [ ]:
!pip install -q "transformers==4.57.6" accelerate
import torch, torchvision, transformers
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| transformers", transformers.__version__, "| GPU:", torch.cuda.is_available())

## 5. Giải nén dữ liệu

In [ ]:
if "OUT" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")
!mkdir -p {BUNDLE_DIR} && unzip -q -o "{DRIVE_DIR}/bundle.zip" -d {BUNDLE_DIR}
print(len(os.listdir(f"{BUNDLE_DIR}/images")), "ảnh;", sorted(os.listdir(BUNDLE_DIR)))

## 6. Chạy thử (khoảng 2–5 phút)

Chấm 3 truy vấn × 20 ứng viên vào tệp tạm, **hai lần**: một lần mã hóa lại mọi ảnh (`--no_vision_cache`,
cách chạy chuẩn của transformers) và một lần dùng lại đặc trưng ảnh đã tính (cách nhanh, mặc định).
Lần đầu sẽ tải Qwen2-VL-2B (khoảng 4.5 GB) từ HuggingFace. Ô này in ra một trong ba kết luận:
- **OK**: hai cách cho cùng điểm, ô 7 dùng cách nhanh.
- **CẢNH BÁO**: hai cách lệch nhau hoặc cách nhanh bị lỗi; ô 7 sẽ tự dùng cách dự phòng (chậm hơn nhưng đúng).
- **LỖI**: cả cách chuẩn cũng không chạy được; đọc thông báo của script ngay phía trên (thường gặp:
  `Non-finite scores` → sửa `DTYPE = "float32"` và `OUT` ở ô 3; `CUDA out of memory` → giảm `BATCH_SIZE`).

Chỉ có 3 truy vấn nên **chưa kết luận được** VLM tốt hay xấu; ô này chỉ kiểm tra mọi thứ chạy đúng.

In [ ]:
if "OUT" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")
from importlib.metadata import PackageNotFoundError, version
try:
    tf_version = version("transformers")
except PackageNotFoundError:
    tf_version = "chưa cài"
if tf_version != "4.57.6":
    raise RuntimeError(f"transformers đang là {tf_version}, chưa phải 4.57.6: "
                       "chạy lại ô 4 rồi chạy lại ô này.")
if not os.path.exists(f"{BUNDLE_DIR}/vlm_rerank.py"):
    raise RuntimeError("Chưa giải nén dữ liệu trong phiên này: chạy lại ô 5 rồi chạy lại ô này.")
import json
!rm -rf /content/smoke
!python {BUNDLE_DIR}/vlm_rerank.py --bundle_dir {BUNDLE_DIR} --out /content/smoke/plain.jsonl \
    --model {MODEL} --top_k 20 --limit 3 --batch_size {BATCH_SIZE} --dtype {DTYPE} --no_vision_cache
!python {BUNDLE_DIR}/vlm_rerank.py --bundle_dir {BUNDLE_DIR} --out /content/smoke/cache.jsonl \
    --model {MODEL} --top_k 20 --limit 3 --batch_size {BATCH_SIZE} --dtype {DTYPE}

def smoke(name):
    path = f"/content/smoke/{name}.jsonl"
    return [json.loads(l) for l in open(path)] if os.path.exists(path) else []

plain, cache = smoke("plain"), smoke("cache")
if len(plain) < 3:
    raise RuntimeError("LỖI: lần chạy chuẩn không hoàn thành; đọc thông báo của vlm_rerank.py ở trên.")
cands = json.load(open(f"{BUNDLE_DIR}/candidates.json"))["queries"]
for r in plain:
    q = cands[r["q"]]
    pos = q["top"][:20].index(q["target"]) if q["target"] in q["top"][:20] else None
    order = sorted(range(20), key=lambda i: -r["scores"][i])
    print(f"Truy vấn {r['q']}: \"{q['text']}\"")
    print("  điểm VLM:", r["scores"])
    print("  ảnh đích không nằm trong top-20 của MagicLens" if pos is None else
          f"  ảnh đích: hạng {pos + 1} theo MagicLens -> hạng {order.index(pos) + 1} theo VLM")
tol = 0.3 if DTYPE == "bfloat16" else 0.1
diff = (max(abs(a - b) for x, y in zip(plain, cache) for a, b in zip(x["scores"], y["scores"]))
        if len(cache) == 3 else float("inf"))
if diff < tol:
    if os.path.exists(NO_CACHE_MARK):
        os.remove(NO_CACHE_MARK)
    print(f"\nOK: hai cách chạy lệch nhau tối đa {diff:.4f}; ô 7 dùng cách nhanh.")
else:
    open(NO_CACHE_MARK, "w").close()
    print("\nCẢNH BÁO: cách nhanh " + ("bị lỗi" if diff == float("inf") else f"lệch {diff:.4f}")
          + "; ô 7 sẽ tự dùng --no_vision_cache (chậm hơn nhưng đúng).")

## 7. Chạy đầy đủ

- 1861 truy vấn × `TOP_K` ứng viên (93 050 cặp với `TOP_K = 50`). Thanh tiến trình hiện tốc độ (pair/s)
  và thời gian còn lại; trên T4 dự kiến vài giờ.
- **Bị ngắt giữa chừng** (hết phiên, mất mạng, đóng tab): kết nối lại, chạy lại ô 1–5 rồi ô này
  (không cần chạy lại ô 6). Script đọc `OUT` trên Drive, bỏ qua các truy vấn đã chấm và chạy tiếp,
  với đúng cách chạy (nhanh hay dự phòng) đã dùng lúc bắt đầu.
- Bản Colab miễn phí giới hạn thời gian dùng GPU; có thể chạy làm nhiều ngày, kết quả vẫn cộng dồn.

In [ ]:
if "OUT" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")
from importlib.metadata import PackageNotFoundError, version
try:
    tf_version = version("transformers")
except PackageNotFoundError:
    tf_version = "chưa cài"
if tf_version != "4.57.6":
    raise RuntimeError(f"transformers đang là {tf_version}, chưa phải 4.57.6: "
                       "chạy lại ô 4 rồi chạy lại ô này.")
if not os.path.exists(f"{BUNDLE_DIR}/vlm_rerank.py"):
    raise RuntimeError("Chưa giải nén dữ liệu trong phiên này: chạy lại ô 5 rồi chạy lại ô này.")
import json
if os.path.exists(OUT + ".meta.json"):  # chạy tiếp: giữ đúng cách chạy lúc bắt đầu
    no_cache = not json.load(open(OUT + ".meta.json")).get("vision_cache", True)
else:
    no_cache = os.path.exists(NO_CACHE_MARK)
CACHE_FLAG = "--no_vision_cache" if no_cache else ""
print("Cách chạy:", "dự phòng (--no_vision_cache)" if no_cache else "nhanh (dùng lại đặc trưng ảnh)")
!python {BUNDLE_DIR}/vlm_rerank.py --bundle_dir {BUNDLE_DIR} --out "{OUT}" \
    --model {MODEL} --top_k {TOP_K} --batch_size {BATCH_SIZE} --dtype {DTYPE} {CACHE_FLAG}

## 8. Kiểm tra kết quả

In [ ]:
if "OUT" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")
import json
n_total = len(json.load(open(f"{BUNDLE_DIR}/candidates.json"))["queries"])
recs = []
for line in open(OUT):
    try:
        recs.append(json.loads(line))
    except json.JSONDecodeError:
        pass  # dòng cuối bị cắt dở; lần chạy ô 7 sau sẽ tự sửa
secs = sum(r["sec"] for r in recs)
n_pairs = sum(len(r["scores"]) for r in recs)
print(f"Đã chấm {len(recs)}/{n_total} truy vấn, {n_pairs} cặp, {secs / 3600:.2f} giờ "
      f"({secs / max(n_pairs, 1) * 1000:.0f} ms/cặp).")
print("XONG: tải kết quả về máy (ô 9)." if len(recs) == n_total
      else "CHƯA XONG: chạy lại ô 7 để chạy tiếp.")

## 9. Tải kết quả về máy

Cần **hai tệp** trên Drive, trong `MyDrive/magiclens_vlm/`: tệp `OUT` (mặc định `vlm_scores.jsonl`)
và tệp `.meta.json` đi kèm (mặc định `vlm_scores.jsonl.meta.json`). Nếu đã đổi `OUT`
(ví dụ `vlm_scores_fp32.jsonl`), hãy tải đúng tên đó.

Tải về bằng giao diện Google Drive, hoặc chạy ô dưới đây (trình duyệt sẽ hỏi lưu hai tệp).
Chép cả hai vào thư mục `vlm_rerank/` của dự án trên máy, rồi chạy (thay tên tệp nếu đã đổi `OUT`):

```bash
.venv/bin/python vlm_rerank_eval.py --scores vlm_rerank/vlm_scores.jsonl
```

In [ ]:
if "OUT" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")
from google.colab import files
files.download(OUT)
files.download(OUT + ".meta.json")